# 概要

sklearnのDigitsデータセットを用いて
3層のSAMACTが手書き数字を学習し、推論するサンプルシナリオ

1. データの準備
2. ニューラルネットワークの構築<br>
  2-1. エンコーダの設定<br>
  2-2. SAMレイヤーの設定<br>
  2-3. デコーダの設定<br>
  2-4. ニューラルネットワークの生成
3. 学習<br>
  3-1. 学習プロパティの設定<br>
  3-2. 学習の実施<br>
  3-3. 学習結果の保存
4. 推論<br>
  4-1. テスト用データセットをまとめて推論<br>
  4-2. 1データを推論


# 追加ライブラリ
scikit-learn >= 1.6.0  
コマンド例: `python3 -m pip install scikit-learn`

# 1.データの準備

In [1]:
import numpy as np
# データセットはsklearnのDigitsを使用。
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

In [2]:
digits = load_digits()
digits.data.shape

(1797, 64)

## ラベル
ラベルは、0から順番にインクリメントされるint型を想定しているので<br>
用意したデータがそうでない場合は、上記のフォーマットに合わせる。今回は修正不要。

In [3]:
digits.target.shape, digits.target.dtype

((1797,), dtype('int64'))

## 正規化
このデータは、0から16までのデータなので、16で割れば正規化できる

In [4]:
normData = digits.data/16.0
normData

array([[0.    , 0.    , 0.3125, ..., 0.    , 0.    , 0.    ],
       [0.    , 0.    , 0.    , ..., 0.625 , 0.    , 0.    ],
       [0.    , 0.    , 0.    , ..., 1.    , 0.5625, 0.    ],
       ...,
       [0.    , 0.    , 0.0625, ..., 0.375 , 0.    , 0.    ],
       [0.    , 0.    , 0.125 , ..., 0.75  , 0.    , 0.    ],
       [0.    , 0.    , 0.625 , ..., 0.75  , 0.0625, 0.    ]])

## 学習用とテスト用に分割

In [5]:
trainData, testData, trainLabel, testLabel = train_test_split(normData, digits.target, test_size=0.2, random_state=0)
trainData.shape, testData.shape, trainLabel.shape, testLabel.shape

((1437, 64), (360, 64), (1437,), (360,))

# 2. ニューラルネットワークの構築
SAMACTのフレームワークの各要素をインスタンスして、ニューラルネットワークを構築する。

本サンプルシナリオでは、3層(64-32-10)のニューラルネットワークを構築する。

In [6]:
# SAMACTの各種機能をimport
from samact import *

## 2-1. エンコーダの設定
エンコーダとして、RateEncodeLayerを指定。<br>
パラメータnUnitsは入力する特徴量の数と一致させる。

In [7]:
inputLayer = RateEncodeLayer(64)
inputLayer

RateEncode(nUnits=64)

## 2-2. SAMレイヤーの設定(隠れ層)

隠れ層として、SAMLayerを指定。

weightDistは下記のプロパティを指定すると安定する。

In [8]:
hiddenLayer = SAMLayer(32, LayerProperty(a=3, p=0.75), Step(), Step(),
                       weightDist=RandomProperty('kaiming', 'normal'))
hiddenLayer

SAMLayer(nUnits=32, gradAct=Step(a=0, b=6, g=18), teacherAct=Step(a=0, b=6, g=18))

## 2-2. SAMレイヤーの設定(出力層)
出力層として、SAMLayerを指定。<br>
パラメータnUnitsは分類するカテゴリ数と一致させる。

In [9]:
outputLayer = SAMLayer(10, LayerProperty(a=3, p=0.75), Step(), Linear(),
                       weightDist=RandomProperty('kaiming', 'normal'))
outputLayer

SAMLayer(nUnits=10, gradAct=Step(a=0, b=6, g=18), teacherAct=Linear(nSlopeLeftShifts=0))

## 2-3. デコーダの設定

デコーダとして、MajorityDecodeLayerを定義。<br>
分類問題に対しては、基本的にこのデコーダを指定する。

In [10]:
decoder = MajorityDecodeLayer()
decoder

MajorityDecode

## 2-4. ネットワークの設定
Sequentialモデルとして、SAMACTのネットワークを定義。

In [11]:
model = Sequential(inputLayer, decoder, [hiddenLayer, outputLayer])
model.Compile(32)
model

Model summary
  Encoder: RateEncode(nUnits=64)
  Layers:
    [0] SAMLayer(nUnits=32, gradAct=Step(a=0, b=6, g=18), teacherAct=Step(a=0, b=6, g=18))
    [1] SAMLayer(nUnits=10, gradAct=Step(a=0, b=6, g=18), teacherAct=Linear(nSlopeLeftShifts=0))
  Decoder: MajorityDecode
  tC: 32

# 3. 学習
学習用のデータセットを利用して、SAMACTを学習させる。

## 3-1. 学習プロパティの設定
学習の際のパラメータを設定する。

In [12]:
learnProperty = LearningProperty(eta=5, iota=2, decayPeriod=1)

## 3-2. 学習の実施
フレームワークのFitメソッドを利用してモデルを学習させる。

In [13]:
fitResult = model.Fit(trainData, trainLabel, 10, learnProperty)

In [14]:
# エポックごとの学習精度
fitResult.metrics

[0.8823938761308281,
 0.9262352122477383,
 0.9457202505219207,
 0.9568545581071677,
 0.9665970772442589,
 0.9631176061238692,
 0.9659011830201809,
 0.9672929714683368,
 0.9665970772442589,
 0.9679888656924147]

## 3-3. 学習結果の保存
学習結果(学習可能パラメータ)をnpzファイルとして保存する。

In [15]:
model.Save('fitResultSample.h5')

# 3-Aux. ファイル読み込みによるモデル復元
3-3で保存したファイルからモデルを復元する。

In [16]:
del model # オブジェクト削除を明示(処理上は冗長)
model = Pretrained('fitResultSample.h5')
model

Model summary
  Encoder: RateEncode(nUnits=64)
  Layers:
    [0] SAMLayer(nUnits=32, gradAct=Step(a=0, b=6, g=18), teacherAct=Step(a=0, b=6, g=18))
    [1] SAMLayer(nUnits=10, gradAct=Step(a=0, b=6, g=18), teacherAct=Linear(nSlopeLeftShifts=0))
  Decoder: MajorityDecode
  tC: 32

# 4. 推論
SAMACTモデルにデータセットを推論させる。

## 4-1. テスト用データセットをまとめて推論
事前にテスト用に温存していたデータセットをSAMACTに推論させる。

In [17]:
evalResult = model.Evaluate(testData, testLabel)
evalResult.metrics

0.9388888888888889

## 4-2. 1データを推論
1つのデータを推論する。(デモアプリなどで利用する想定)

In [18]:
predict = model.Predict(testData[0])
f'prediction is {predict}, golden is {testLabel[0]}'

'prediction is 2, golden is 2'